In [1]:
# Sugar Trap: Market Gap Analysis
# Amalitech Data Engineering Project Assessment

"""
Objective:
Identify potential nutritional gaps in the snack market by
analyzing product categories, sugar, protein, and ingredients.
"""

# 1. Environment setup
!pip install -q pandas numpy matplotlib seaborn pyarrow

In [2]:
# 2. Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# 3. Configuration

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

print("Environment successfully initialized.")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Environment successfully initialized.
Pandas version: 2.2.3
NumPy version: 2.4.6


## 4. Data Acquisition

### Data Source
Open Food Facts – Global Food Products Database.

We will work with a sample of up to 500,000 records and
select only the fields required for the analysis.

The source dataset is tab-separated despite its `.csv`
extension.

In [3]:
# 4.1 Data source and configuration

DATA_URL = (
    "https://static.openfoodfacts.org/data/"
    "en.openfoodfacts.org.products.csv"
)

REQUIRED_COLUMNS = [
    "code",
    "product_name",
    "categories_tags",
    "sugars_100g",
    "proteins_100g",
    "fat_100g",
    "fiber_100g",
    "ingredients_text"
]

SAMPLE_SIZE = 500_000
CHUNK_SIZE = 25_000

print("Data source configured.")

Data source configured.


### Column Selection

We select only the variables relevant to nutritional
analysis, product classification and ingredient analysis.

In [4]:
# 4.2 Stream data in manageable chunks

response = requests.get(
    DATA_URL,
    stream=True,
    headers={
        "User-Agent": "SugarTrapAnalysis/1.0"
    },
    timeout=(30, 300)
)

response.raise_for_status()

response.raw.decode_content = True

chunks = []
rows_collected = 0

try:
    reader = pd.read_csv(
        response.raw,
        sep="\t",
        usecols=REQUIRED_COLUMNS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:

        remaining = SAMPLE_SIZE - rows_collected

        if len(chunk) > remaining:
            chunk = chunk.iloc[:remaining]

        chunks.append(chunk)
        rows_collected += len(chunk)

        print(f"Rows collected: {rows_collected:,}")

        if rows_collected >= SAMPLE_SIZE:
            break

finally:
    response.close()

raw_df = pd.concat(chunks, ignore_index=True)

del chunks

print("\nDataset successfully loaded!")
print("Shape:", raw_df.shape)

display(raw_df.head())

Rows collected: 25,000
Rows collected: 50,000
Rows collected: 75,000
Rows collected: 100,000
Rows collected: 125,000
Rows collected: 150,000
Rows collected: 175,000
Rows collected: 200,000
Rows collected: 225,000
Rows collected: 250,000
Rows collected: 275,000
Rows collected: 300,000
Rows collected: 325,000
Rows collected: 350,000
Rows collected: 375,000
Rows collected: 400,000
Rows collected: 425,000
Rows collected: 450,000
Rows collected: 475,000
Rows collected: 500,000

Dataset successfully loaded!
Shape: (500000, 8)


,code,product_name,categories_tags,ingredients_text,fat_100g,sugars_100g,fiber_100g,proteins_100g
0,NaN,NaN,NaN,"en:to-be-completed,en:nutrition-facts-complete...",NaN,NaN,NaN,NaN
1,NaN,"en:palm-oil-free,en:vegan-status-unknown,en:ve...","en:e322,en:e322i","en:fat-in-high-quantity,en:saturated-fat-in-hi...",NaN,NaN,NaN,NaN
2,NaN,"en:palm-oil-free,en:non-vegan,en:vegetarian","en:e322,en:e322i,en:e330,en:e407,en:e412,en:e4...","en:fat-in-moderate-quantity,en:saturated-fat-i...",0.378651,NaN,NaN,NaN
3,NaN,"en:palm-oil-free,en:non-vegan,en:vegetarian-st...","en:e270,en:e322,en:e322i,en:e407,en:e412,en:e4...","en:fat-in-moderate-quantity,en:saturated-fat-i...",2.278935,NaN,NaN,NaN
4,NaN,"en:palm-oil-free,en:non-vegan,en:vegetarian","en:e322,en:e322i,en:e330,en:e407,en:e412,en:e4...","en:fat-in-moderate-quantity,en:saturated-fat-i...",NaN,NaN,NaN,NaN


In [5]:
# 4.4 Initial inspection

print("Dataset dimensions:", raw_df.shape)

print("\nData types:")
display(raw_df.dtypes)

print("\nMissing values:")
display(raw_df.isna().sum().sort_values(ascending=False))

print("\nDataset information:")
raw_df.info()

Dataset dimensions: (500000, 8)

Data types:


,0
code,object
product_name,object
categories_tags,object
ingredients_text,object
fat_100g,float64
sugars_100g,float64
fiber_100g,float64
proteins_100g,float64



Missing values:


,0
fiber_100g,400951
sugars_100g,371096
fat_100g,365146
proteins_100g,365121
ingredients_text,252346
categories_tags,249174
product_name,17790
code,4205



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   code              495795 non-null  object 
 1   product_name      482210 non-null  object 
 2   categories_tags   250826 non-null  object 
 3   ingredients_text  247654 non-null  object 
 4   fat_100g          134854 non-null  float64
 5   sugars_100g       128904 non-null  float64
 6   fiber_100g        99049 non-null   float64
 7   proteins_100g     134879 non-null  float64
dtypes: float64(4), object(4)
memory usage: 30.5+ MB
